In [ ]:
from pypot.dynamixel import DxlIO
from pypot.dynamixel.protocol.v1 import *

from glob import glob

ports = glob('/dev/ttyACM*')
assert len(ports) == 1

port = ports[0]
print('Connecting on port: {}'.format(port))
dxl_io = DxlIO(port)

poulpe_id = 42
N_AXIS = 2

In [ ]:
ping_packet = DxlPingPacket(poulpe_id)
dxl_io._send_packet(ping_packet)

In [ ]:
import struct

def read_current_pos():
    pos_packet = DxlReadDataPacket(poulpe_id, 50, N_AXIS * 4)
    resp = dxl_io._send_packet(pos_packet)
    data = bytearray(resp.parameters)
    pos = struct.unpack(N_AXIS * 'f', data)
    return pos

read_current_pos()

In [ ]:
import struct

def read_target_position():
    pos_packet = DxlReadDataPacket(poulpe_id, 60, N_AXIS * 4)
    resp = dxl_io._send_packet(pos_packet,wait_for_status_packet=True)
    data = bytearray(resp.parameters)
    pos = struct.unpack(N_AXIS * 'f', data)
    return pos

read_target_position()

In [ ]:
def write_target_position(target):
    p = DxlWriteDataPacket(poulpe_id, 60, struct.pack(N_AXIS * 'f', *target))
    resp = dxl_io._send_packet(p,wait_for_status_packet=True)
    return resp

write_target_position([0.5, 0.5])

In [ ]:
def read_torque_enabled():
    p = DxlReadDataPacket(poulpe_id, 40, N_AXIS)
    resp = dxl_io._send_packet(p)
    data = bytearray(resp.parameters)
    torque = struct.unpack(N_AXIS * '?', data)
    return torque

read_torque_enabled()

In [ ]:
def write_torque_enabled(torque):
    p = DxlWriteDataPacket(poulpe_id, 40, struct.pack(N_AXIS * '?', *torque))
    resp = dxl_io._send_packet(p)
    return resp

write_torque_enabled([False, False])

In [ ]:
write_torque_enabled([True, True])

In [ ]:
import time
import numpy as np

pos = []
send_target = []
read_target = []


t0 = time.time()
while True:
    if time.time() - t0 > 5:
        break

    target = [
        30 * np.sin(2 * np.pi * 0.5 * (time.time()-t0)), 
        30 * np.sin(2 * np.pi * 0.5 * (time.time()-t0)),
    ]
    write_target_position(target)
    
    send_target.append(target)
    time.sleep(0.001)
    cur=read_current_pos()
    pos.append(cur)
    read_target.append(cur)
    time.sleep(0.001)

In [ ]:
np.array(send_target) - np.array(read_target)